# Aula 08 — Laboratório
## Quanto o modelo erra de verdade

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Os modelos das últimas semanas, avaliados com cuidado. Neste laboratório:

1. a nota de um teste único depende do sorteio; a validação cruzada;
2. seis modelos — logit, Lasso, componentes + logit, árvore, floresta e boosting — nas
   mesmas dobras;
3. calibração: a probabilidade prevista bate com a observada? E quanto isso vale?
4. vazamento de propósito: colunas escolhidas antes de dividir e uma coluna do futuro;
5. tempo e painel: três validações num painel simulado de municípios, contra o erro de
   verdade.

No fim, o **laudo de validação**: a tabela dos seis modelos e a recomendação de um deles.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import io, os, sys, zipfile, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Laranja = logit (e o modelo linear do painel),
# azul = boosting, verde = floresta, cinza = os demais. Bordo = destaque.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#8A8D96"

try:
    import sklearn
    from sklearn import __version__ as VERSAO_SKLEARN
    from sklearn.decomposition import PCA
    from sklearn.ensemble import (HistGradientBoostingClassifier,
                                  HistGradientBoostingRegressor, RandomForestClassifier)
    from sklearn.feature_selection import SelectKBest, f_classif
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.metrics import brier_score_loss, roc_auc_score
    from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold,
                                         StratifiedKFold, train_test_split)
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import FunctionTransformer, StandardScaler
    from sklearn.tree import DecisionTreeClassifier
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")

print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("\nAmbiente pronto.")

## Bloco 1 — um teste é um sorteio

Na Aula 05, os modelos foram comparados numa divisão só: 75% dos clientes para ajustar,
25% para testar, sorteados com a semente 2026. Aqui, a mesma base e os mesmos modelos,
com outras divisões.

In [ ]:
# Bloco 1 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (frequencias de limite, idade, sexo, escolaridade e
# estado civil; como a situacao do cartao passa de um mes para o outro; uso do
# limite e fracao da fatura paga) e um logit simples do calote. Serve so para
# gerar clientes inventados com a mesma estrutura.

ALVO = 'default.payment.next.month'
ATRASOS = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
FATURAS = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
PAGAMENTOS = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
COLUNAS = ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']
ESTADOS = [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]

ESP_TAIWAN = {'LIMIT_BAL': [[10000, 16000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000,
                110000, 120000, 130000, 140000, 150000, 160000, 170000, 180000, 190000, 200000,
                210000, 220000, 230000, 240000, 250000, 260000, 270000, 280000, 290000, 300000,
                310000, 320000, 327680, 330000, 340000, 350000, 360000, 370000, 380000, 390000,
                400000, 410000, 420000, 430000, 440000, 450000, 460000, 470000, 480000, 490000,
                500000, 510000, 520000, 530000, 540000, 550000, 560000, 570000, 580000, 590000,
                600000, 610000, 620000, 630000, 640000, 650000, 660000, 670000, 680000, 690000,
                700000, 710000, 720000, 730000, 740000, 750000, 760000, 780000, 800000, 1000000],
               [0.01643, 7e-05, 0.06587, 0.05367, 0.00767, 0.11217, 0.0275, 0.02437, 0.05223,
                0.0217, 0.03493, 0.0196, 0.0242, 0.0243, 0.02497, 0.037, 0.02313, 0.01773, 0.03317,
                0.00763, 0.05093, 0.02433, 0.01563, 0.02457, 0.02063, 0.01167, 0.01737, 0.00793,
                0.01643, 0.0116, 0.01847, 0.00907, 0.0104, 3e-05, 0.00577, 0.00723, 0.0077, 0.02937,
                0.00237, 0.0052, 0.0058, 0.00903, 0.0026, 0.0056, 0.00277, 0.00277, 0.00537,
                0.00267, 0.00267, 0.00263, 0.00213, 0.02407, 0.00063, 0.00067, 0.00033, 0.0002,
                0.0007, 0.00033, 0.00027, 0.00037, 0.0002, 0.00053, 0.00037, 0.0003, 0.00023,
                0.00023, 0.0001, 0.0001, 0.0001, 0.00013, 3e-05, 0.00027, 0.0002, 0.0001, 7e-05,
                7e-05, 0.00013, 3e-05, 7e-05, 7e-05, 3e-05]],
 'AGE': [[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
          43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64,
          65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 79],
         [0.00223, 0.01867, 0.03103, 0.03757, 0.03953, 0.04187, 0.04923, 0.04697, 0.0535, 0.0465,
          0.04057, 0.0386, 0.0382, 0.03873, 0.0371, 0.03693, 0.0347, 0.03147, 0.0318, 0.029,
          0.02747, 0.02647, 0.02233, 0.02333, 0.02057, 0.019, 0.0167, 0.01553, 0.01507, 0.0137,
          0.01133, 0.01013, 0.01083, 0.00823, 0.00697, 0.00593, 0.00407, 0.00407, 0.00277, 0.00223,
          0.00187, 0.00147, 0.00103, 0.00103, 0.0008, 0.00083, 0.00053, 0.00017, 0.0005, 0.00033,
          0.0001, 0.0001, 0.00013, 3e-05, 0.0001, 3e-05]],
 'SEX': [[1, 2], [0.39627, 0.60373]],
 'EDUCATION': [[0, 1, 2, 3, 4, 5, 6], [0.00047, 0.35283, 0.46767, 0.1639, 0.0041, 0.00933, 0.0017]],
 'MARRIAGE': [[0, 1, 2, 3], [0.0018, 0.4553, 0.53213, 0.01077]],
 'inicio': [0.16317, 0.19133, 0.54287, 0.0, 0.0922, 0.00613, 0.00163, 0.00043, 0.00063, 0.00153,
            7e-05],
 'transicao': [[0.8128, 0.0682, 0.05038, 0.05694, 0.01168, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.06576, 0.75689, 0.12005, 0.02145, 0.03585, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [5e-05, 0.05058, 0.88872, 7e-05, 0.06058, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.00029, 0.00029, 0.00029, 0.99707, 0.00029, 0.00029, 0.00029, 0.00029, 0.00029,
                0.00029, 0.00029],
               [0.00061, 0.08014, 0.17267, 0.10284, 0.58047, 0.06326, 0.0, 0.0, 0.0, 0.0, 0.0],
               [1e-05, 0.07581, 0.08303, 0.09837, 0.32669, 0.15884, 0.2572, 1e-05, 1e-05, 1e-05,
                1e-05],
               [3e-05, 0.02124, 0.02124, 0.08488, 0.22542, 0.07693, 0.28111, 0.28907, 3e-05, 3e-05,
                3e-05],
               [9e-05, 0.02709, 0.02709, 0.06309, 0.16209, 0.06309, 0.09909, 0.10809, 0.45009,
                9e-05, 9e-05],
               [0.00016, 0.016, 0.016, 0.03185, 0.07939, 0.016, 0.016, 0.04769, 0.06354, 0.7132,
                0.00016],
               [5e-05, 5e-05, 5e-05, 0.00483, 0.26785, 0.00961, 5e-05, 5e-05, 0.00483, 0.6026,
                0.11004],
               [0.0011, 0.0011, 0.0011, 0.11087, 0.33041, 0.0011, 0.11087, 0.0011, 0.0011, 0.11087,
                0.33041]],
 'uso': {'-2': [0.0, 0.0933],
         '-1': [0.0136, 0.1606],
         '0': [0.5141, 0.3382],
         '1': [0.0313, 0.4004],
         '2': [0.6213, 0.3567],
         '3': [0.7173, 0.3796],
         '4': [0.8591, 0.3797],
         '5': [0.9343, 0.3888],
         '6': [0.9898, 0.4309],
         '7': [0.0808, 0.374],
         '8': [1.0963, 0.3876]},
 'pago': {'-2': 1.0,
          '-1': 1.0,
          '0': 0.0491,
          '1': 0.0375,
          '2': 0.0433,
          '3': 0.0,
          '4': 0.0,
          '5': 0.0,
          '6': 0.0,
          '7': 0.0,
          '8': 0.0},
 'calote': [1.59163, -0.14007, -0.37803, 0.57321, 1.95988, 1.97565, 1.37242, 0.5472, -0.29859,
            0.00842]}


def logistica(z):
    return 1 / (1 + np.exp(-z))


def taiwan_sintetica(ESP, n=30000, semente=2005):
    """Plano B: clientes INVENTADOS com as mesmas colunas, os mesmos tipos e
    os mesmos códigos da base real. A situação de cada mês segue a do mês
    anterior; a fatura acompanha o limite e a situação; o calote sai de um
    logit da situação em setembro e em agosto, do limite e da idade."""
    rng = np.random.default_rng(semente)

    def sorteio(chave, k):
        v, p = ESP[chave]
        p = np.array(p, dtype=float)
        return rng.choice(np.array(v), k, p=p / p.sum())

    d = {"ID": np.arange(1, n + 1)}
    for c in ("LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"):
        d[c] = sorteio(c, n)
    ini = np.array(ESP["inicio"], dtype=float)
    T = np.array(ESP["transicao"], dtype=float)
    sit = np.zeros((n, 6), dtype=int)                       # abr → set
    sit[:, 0] = rng.choice(ESTADOS, n, p=ini / ini.sum())
    for t in range(1, 6):
        u = rng.random(n)
        acum = np.cumsum(T[sit[:, t - 1] + 2], axis=1)
        sit[:, t] = np.array(ESTADOS)[(u[:, None] > acum).sum(axis=1).clip(0, 10)]
    for j, c in enumerate(ATRASOS):                          # PAY_0 = setembro
        d[c] = sit[:, 5 - j]
    lim = d["LIMIT_BAL"].astype(float)
    for j, c in enumerate(FATURAS):
        e = sit[:, 5 - j]
        med = np.array([ESP["uso"][str(v)][0] for v in e])
        dp = np.array([ESP["uso"][str(v)][1] for v in e])
        uso = np.clip(med + dp * rng.normal(0, 1, n), -0.05, 1.3)
        uso = np.where(e == -2, np.where(rng.random(n) < 0.7, 0, uso * 0.1), uso)
        d[c] = np.round(lim * uso).astype(int)
    for j, c in enumerate(PAGAMENTOS):
        if j < 5:
            e = sit[:, 5 - j]
            fr = np.array([ESP["pago"][str(v)] for v in e])
            base = np.maximum(d[FATURAS[j + 1]], 0)
            d[c] = np.round(base * fr * np.exp(rng.normal(0, 0.5, n))).astype(int)
        else:
            d[c] = np.round(np.maximum(d[FATURAS[5]], 0) * 0.1).astype(int)
    b = np.array(ESP["calote"])
    A = np.column_stack([np.ones(n)]
                        + [(np.clip(d["PAY_0"], -2, 4) == e).astype(float) for e in range(-2, 5) if e != -1]
                        + [(d["PAY_2"] >= 2).astype(float), np.log(lim), d["AGE"]])
    d[ALVO] = (rng.random(n) < logistica(A @ b)).astype(int)
    return pd.DataFrame(d)[COLUNAS]


print("Plano B pronto (so e usado se a base real nao carregar).")

In [ ]:
# Bloco 1 — os cartoes de credito de Taiwan (Yeh e Lien, 2009), os da Aula 05.
#
# Tres caminhos, nesta ordem:
#   1. o arquivo credito_taiwan.csv, na pasta Dados do repositorio;
#   2. a planilha original da UCI (zip com .xls; precisa do pacote xlrd);
#   3. o pacote `ucimlrepo`, se estiver instalado.
# Se os tres falharem, segue o PLANO B (celula acima), com dados inventados.

ARQUIVO = "../Dados/credito_taiwan.csv"
URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"


def padronizar(d):
    """Mesmas colunas, na mesma ordem, venha a base de onde vier."""
    d = d.rename(columns={"default payment next month": ALVO})
    if list(d.columns) != COLUNAS:
        if len(d.columns) == len(COLUNAS) - 1:          # veio sem a coluna de ID
            d.insert(0, "ID", np.arange(1, len(d) + 1))
        if len(d.columns) == len(COLUNAS):              # X1..X23, Y -> nomes da UCI
            d.columns = COLUNAS
    if list(d.columns) != COLUNAS or len(d) < 1000:
        raise ValueError("a tabela nao tem a estrutura esperada")
    return d.astype(int)


def ler_planilha_uci(conteudo):
    with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
        nome = [n for n in z.namelist() if n.endswith(".xls")][0]
        return pd.read_excel(io.BytesIO(z.read(nome)), header=1)


def carregar_taiwan(arquivo=ARQUIVO, url=URL, timeout=120):
    if os.path.exists(arquivo):
        return padronizar(pd.read_csv(arquivo)), "arquivo local " + arquivo
    try:
        import requests
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return padronizar(ler_planilha_uci(r.content)), "UCI (download)"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    try:
        from ucimlrepo import fetch_ucirepo
        return padronizar(fetch_ucirepo(id=350).data.original), "pacote ucimlrepo"
    except Exception:
        pass
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Mesmas colunas, mesmos")
    print("tipos e mesmos codigos da base real; clientes e relacoes sao")
    print("sinteticos. Os numeros NAO valem como resultado sobre Taiwan.")
    print("=" * 70)
    return taiwan_sintetica(ESP_TAIWAN), "PLANO B (sintetico)"


SEMENTE_DIVISAO = 2026


def preparar_taiwan(dados):
    """Da tabela crua à matriz do modelo.

    * sexo e estado civil ficam FORA do modelo;
    * escolaridade: pós-graduação (referência), universidade, ensino médio e
      "outros" (códigos 0, 4, 5 e 6, que a documentação não distingue);
    * atrasos de abril a setembro como estão: −2, −1 e 0 indicam conta em dia;
      1 a 8, meses de atraso; 9, nove meses ou mais;
    * treino e teste: 75% e 25% dos clientes, sorteados com a mesma proporção
      de calote nos dois (todos são do mesmo mês: não há corte por data).
    """
    d = dados.copy()
    y = d[ALVO].to_numpy().astype(int)
    esc = d["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
    X = d[["LIMIT_BAL", "AGE"] + ATRASOS + FATURAS + PAGAMENTOS].astype(float)
    X["EDU_universidade"] = (esc == 2).astype(float)
    X["EDU_medio"] = (esc == 3).astype(float)
    X["EDU_outros"] = (esc == 4).astype(float)
    treino = np.zeros(len(y), dtype=bool)
    idx_tr, _ = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y,
                                 random_state=SEMENTE_DIVISAO)
    treino[idx_tr] = True
    return X, y, treino


cartoes, FONTE = carregar_taiwan()
X, y, treino = preparar_taiwan(cartoes)
print(f"fonte: {FONTE}")
print(f"{X.shape[0]} clientes, {X.shape[1]} colunas no modelo, {100 * y.mean():.1f}% de calote")

In [ ]:
# Bloco 1 — as ferramentas: os seis modelos e as metricas. Leia, nao precisa
# alterar. Cada modelo e uma "fabrica": uma funcao que devolve o modelo novo,
# ainda sem ajustar.

SEMENTE = 2026

def faixas_de_atraso(X):
    """Cada coluna de atraso vira três colunas 0/1 — um mês, dois meses, três
    ou mais —, com "em dia" como referência. É o degrau escrito à mão, para o
    logit poder vê-lo."""
    Z = X.copy()
    for c in ATRASOS:
        v = Z.pop(c)
        Z[c + "_1m"] = (v == 1).astype(float)
        Z[c + "_2m"] = (v == 2).astype(float)
        Z[c + "_3m+"] = (v >= 3).astype(float)
    return Z


def expandir(X):
    """As colunas do Lasso: cada atraso vira três colunas 0/1 (um mês, dois,
    três ou mais), faturas e pagamentos em log, o uso do limite em cada mês e
    a fração da fatura anterior que foi paga. São 46 colunas no lugar de 23."""
    Z = faixas_de_atraso(X)
    for c in FATURAS:
        Z[c] = np.sign(X[c]) * np.log1p(np.abs(X[c]))
    for c in PAGAMENTOS:
        Z[c] = np.log1p(X[c].clip(lower=0))
    Z["LIMIT_BAL"] = np.log(X["LIMIT_BAL"])
    for i, c in enumerate(FATURAS):
        Z[f"uso_{i + 1}"] = X[c] / X["LIMIT_BAL"]
    for i in range(5):
        Z[f"pago_{i + 1}"] = (X[PAGAMENTOS[i]]
                              / np.maximum(X[FATURAS[i + 1]], 1)).clip(0, 2)
    return Z


def logit_l1(C=1.0):
    """Logit com penalidade L1 (o Lasso da Semana 5, para classificação). A
    forma de pedir L1 mudou no scikit-learn 1.8: antes, penalty="l1"; depois,
    l1_ratio=1. O liblinear sorteia a ordem das observações: semente fixa."""
    versao = tuple(int(x) for x in VERSAO_SKLEARN.split(".")[:2])
    if versao >= (1, 8):
        return LogisticRegression(l1_ratio=1, C=C, solver="liblinear", max_iter=5000,
                                  random_state=0)
    return LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=5000,
                              random_state=0)


def modelo_logit():
    """Logit com as 23 colunas como estão (Aula 05)."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))


def modelo_lasso():
    """Lasso nas 46 colunas de `expandir`. O C (o inverso do λ) é escolhido
    por validação cruzada DENTRO do treino de cada dobra."""
    pipe = make_pipeline(FunctionTransformer(expandir), StandardScaler(), logit_l1())
    return GridSearchCV(pipe, {"logisticregression__C": list(np.logspace(-3, 0, 7))},
                        scoring="neg_log_loss", cv=3)


def modelo_componentes():
    """Componentes principais (Semana 8) + logit: as 23 colunas padronizadas
    resumidas em 5 componentes, que entram no logit."""
    return make_pipeline(StandardScaler(), PCA(5), LogisticRegression(max_iter=5000))


def modelo_arvore():
    """Árvore com profundidade e tamanho mínimo de folha escolhidos por
    validação cruzada DENTRO do treino de cada dobra."""
    return GridSearchCV(DecisionTreeClassifier(random_state=0),
                        {"max_depth": [2, 3, 4, 5, 6, 7, 8], "min_samples_leaf": [20, 50, 100]},
                        scoring="roc_auc", cv=3)


def modelo_floresta():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features="sqrt",
                                  random_state=0, n_jobs=-1)


def modelo_boosting():
    """Boosting na versão rápida do scikit-learn, que agrupa os valores de
    cada coluna em faixas antes de procurar os cortes."""
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3,
                                          random_state=0)


MODELOS = {
    "logit": modelo_logit,
    "lasso": modelo_lasso,
    "componentes": modelo_componentes,
    "arvore": modelo_arvore,
    "floresta": modelo_floresta,
    "boosting": modelo_boosting,
}


def auc(y, p):
    return float(roc_auc_score(y, p))


def brier(y, p):
    """Erro quadrático médio da probabilidade: a média de (p − y)²."""
    return float(brier_score_loss(y, p))


def dobras(y, k=5, semente=SEMENTE):
    """Cinco dobras com a mesma proporção de calote em cada uma."""
    return list(StratifiedKFold(k, shuffle=True, random_state=semente).split(np.zeros(len(y)), y))


def nota_no_teste(fabrica, X, y, semente):
    """AUC no teste de uma divisão 75/25 sorteada com a semente dada."""
    tr, te = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y,
                              random_state=semente)
    m = fabrica().fit(X.iloc[tr], y[tr])
    return auc(y[te], m.predict_proba(X.iloc[te])[:, 1])


print("Ferramentas prontas:", ", ".join(MODELOS))

In [ ]:
# Bloco 1 — vinte sorteios. COMPLETE.
#
# Para cada semente de 0 a 19, calcule a AUC no teste do logit e do boosting
# com nota_no_teste(fabrica, X, y, semente). Depois, a divisao da Aula 05.

notas = {"logit": [], "boosting": []}
for semente in range(20):
    for nome in notas:
        # COMPLETE
        notas[nome].append(___)

aula05 = {nome: nota_no_teste(MODELOS[nome], X, y, SEMENTE) for nome in notas}
tabela = pd.DataFrame(notas)
print(tabela.describe().loc[["min", "50%", "max"]].round(3))
print("divisao da Aula 05:", {k: round(v, 3) for k, v in aula05.items()})

fig, ax = plt.subplots(figsize=(10, 3.2))
for i, (nome, cor) in enumerate((("logit", LARANJA), ("boosting", AZUL))):
    ax.scatter(tabela[nome], np.full(len(tabela), i), color=cor, alpha=0.6, s=40)
    ax.scatter([aula05[nome]], [i], color=BORDO, marker="D", s=90, zorder=3)
ax.set_yticks([0, 1]); ax.set_yticklabels(["logit", "boosting"])
ax.set_xlabel("AUC no teste (losango: a divisao da Aula 05)")
plt.show()

In [ ]:
# Bloco 1 — cinco dobras. COMPLETE.
#
# Validacao cruzada estratificada do boosting: para cada dobra, ajuste nas
# outras quatro e calcule a AUC na que ficou de fora.

DOBRAS = dobras(y)                 # 5 dobras, semente 2026
notas_boosting = []
for treino_idx, teste_idx in DOBRAS:
    # COMPLETE
    modelo = ___
    p = ___
    notas_boosting.append(auc(y[teste_idx], p))

notas_boosting = np.array(notas_boosting)
print("AUC por dobra:", notas_boosting.round(3))
print(f"media {notas_boosting.mean():.3f} | desvio-padrao {notas_boosting.std(ddof=1):.3f}")

### Responda

> Quanto a nota de um teste único varia de um sorteio para outro? O que a validação
> cruzada entrega que um teste único não entrega?

## Bloco 2 — seis modelos, as mesmas dobras

Os seis modelos nas mesmas cinco dobras. A função `validar` ajusta cada modelo em cada
dobra e guarda a probabilidade prevista para os clientes que ficaram de fora: no fim, cada
cliente tem uma previsão de cada modelo, feita por um modelo que não o viu.

In [ ]:
# Bloco 2 — a validacao. COMPLETE. (Leva um ou dois minutos.)

def validar(fabricas, X, y, divisoes):
    """Ajusta cada modelo em cada dobra e prevê os clientes que ficaram de
    fora. Devolve as probabilidades fora da amostra (cada cliente previsto
    uma vez, por um modelo que não o viu) e as notas de cada dobra."""
    prob = {n: np.zeros(len(y)) for n in fabricas}
    notas = {n: [] for n in fabricas}
    for treino, teste in divisoes:
        for n, fabrica in fabricas.items():
            m = fabrica().fit(X.iloc[treino], y[treino])
            p = m.predict_proba(X.iloc[teste])[:, 1]
            prob[n][teste] = p
            notas[n].append({"auc": auc(y[teste], p), "brier": brier(y[teste], p),
                             "acuracia": float(((p >= 0.5) == y[teste]).mean())})
    return prob, {n: pd.DataFrame(v) for n, v in notas.items()}


# COMPLETE: os seis modelos (MODELOS) nas DOBRAS do Bloco 1
prob, notas_dobra = ___
print("pronto:", list(prob))

In [ ]:
# Bloco 2 — a tabela. COMPLETE.
#
# Para cada modelo: AUC e Brier com todas as previsoes fora da amostra, e a
# acuracia com corte em 50%. Na ultima linha, a referencia: dar a todos a
# mesma probabilidade, a taxa de calote.

linhas = {}
for nome, p in prob.items():
    # COMPLETE
    linhas[nome] = {"auc": ___, "brier": ___, "acuracia": ___}
linhas["todos com a taxa media"] = {"auc": 0.5, "brier": brier(y, np.full(len(y), y.mean())),
                                    "acuracia": 1 - y.mean()}
resumo = pd.DataFrame(linhas).T.sort_values("auc", ascending=False)
print(resumo.round(4))

In [ ]:
# Bloco 2 — as diferencas, dobra a dobra. COMPLETE.
#
# notas_dobra[nome] e uma tabela com a AUC de cada dobra. Calcule, dobra a
# dobra, boosting - logit e floresta - boosting.

# COMPLETE
dif_bl = ___
dif_fb = ___

print("boosting - logit  :", np.round(dif_bl, 4), "| a favor em", (dif_bl > 0).sum(), "de 5")
print("floresta - boosting:", np.round(dif_fb, 4), "| a favor em", (dif_fb > 0).sum(), "de 5")
print(f"variacao de uma dobra para outra (desvio da AUC do boosting): "
      f"{notas_dobra['boosting']['auc'].std(ddof=1):.4f}")

### ✔ CHECKPOINT I

Nas cinco dobras, a floresta tem AUC um pouco maior que a do boosting e fica à frente dele
dobra após dobra.

Em duas frases: o que a validação autoriza a dizer sobre os dois? Se tivesse de escolher
um para o banco, o que você olharia além da AUC?

## Bloco 3 — probabilidade que se possa usar

A AUC mede se o modelo **ordena** bem os clientes. Provisão para perdas, preço do crédito e
a regra de limiar da Aula 05 usam a **probabilidade** em si. Um modelo calibrado acerta as
duas coisas: dos clientes a quem ele dá 30%, uns 30% dão calote.

In [ ]:
# Bloco 3 — calibracao. COMPLETE.

def calibracao(p, y, n=10):
    """Clientes em n grupos de igual tamanho pela probabilidade prevista;
    em cada um, a média prevista e a taxa de calote observada."""
    g = pd.qcut(pd.Series(p).rank(method="first"), n, labels=False)
    d = pd.DataFrame({"prevista": p, "observada": y, "grupo": g})
    return d.groupby("grupo")[["prevista", "observada"]].mean()


# COMPLETE: a calibracao do logit e a do boosting, com as previsoes fora da amostra
cal_logit = ___
cal_boost = ___

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 0.8], [0, 0.8], "--", color=BORDO, label="previsto = observado")
ax.plot(cal_logit["prevista"], cal_logit["observada"], "o-", color=LARANJA, label="logit")
ax.plot(cal_boost["prevista"], cal_boost["observada"], "o-", color=AZUL, label="boosting")
ax.set_xlabel("probabilidade prevista (media do decimo)")
ax.set_ylabel("taxa de calote observada")
ax.legend(); plt.show()
print(pd.concat({"logit": cal_logit, "boosting": cal_boost}, axis=1).round(3))

In [ ]:
# Bloco 3 — o lucro. COMPLETE.
#
# lucro_por_limiar (Aula 05) devolve o lucro por 1.000 clientes para cada
# limiar de 0 a 1: aprova quem tem probabilidade prevista ate o limiar. Com
# margem 1 e perda 4, a regra diz: aprove abaixo de 20%.

MARGEM, PERDA = 1.0, 4.0
LIMIARES = np.round(np.linspace(0, 1, 101), 2)

def lucro_por_limiar(p, y, margem=MARGEM, perda=PERDA, limiares=LIMIARES):
    """Lucro da carteira aprovada (quem tem probabilidade prevista até o
    limiar), em unidades de margem por 1.000 clientes avaliados."""
    p, y = np.asarray(p), np.asarray(y)
    saida = []
    for t in limiares:
        apr = p <= t
        saida.append(1000 * (margem * ((y == 0) & apr).sum()
                             - perda * ((y == 1) & apr).sum()) / len(y))
    return np.array(saida)


lucro = {}
for nome, p in prob.items():
    curva = lucro_por_limiar(p, y)
    # COMPLETE: o lucro no limiar de 20%, o maximo e o limiar do maximo
    lucro[nome] = {"regra_20": ___, "maximo": ___, "limiar_do_maximo": ___}
print(pd.DataFrame(lucro).T.round(2))

## Bloco 4 — vazamento de propósito

Duas experiências em que o teste "vê" o que não deveria.

**A peneira.** 100 clientes de Taiwan, com o calote verdadeiro, e 5.000 colunas
sorteadas — ruído puro. Escolhem-se as 20 colunas mais ligadas ao calote e
valida-se um logit com elas. Nenhuma coluna tem informação: a AUC honesta é 0,5.

In [ ]:
# Bloco 4 — a peneira. COMPLETE.

CLIENTES_PENEIRA = 100
COLUNAS_ESCOLHIDAS = 20


def amostra_peneira(n_ruido, semente, n=CLIENTES_PENEIRA):
    """`n` clientes, com o calote verdadeiro, e `n_ruido` colunas de ruido."""
    idx, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y,
                              random_state=semente)
    R = np.random.default_rng(semente).normal(size=(n, n_ruido))
    return R, y[idx]


def peneira_errada(R, y, k=COLUNAS_ESCOLHIDAS, semente=0):
    """ERRADO: escolhe as k colunas mais ligadas ao calote olhando os n
    clientes, e só depois valida."""
    Rk = SelectKBest(f_classif, k=min(k, R.shape[1])).fit(R, y).transform(R)
    p = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=semente).split(Rk, y):
        p[te] = LogisticRegression(max_iter=3000).fit(Rk[tr], y[tr]).predict_proba(Rk[te])[:, 1]
    return auc(y, p)


def peneira_certa(R, y, k=COLUNAS_ESCOLHIDAS, semente=0):
    """CERTO: a escolha das colunas faz parte do modelo e e refeita dentro
    do treino de cada dobra."""
    # COMPLETE: um pipeline com SelectKBest(f_classif, k=...) e LogisticRegression
    modelo = ___
    p = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=semente).split(R, y):
        p[te] = modelo.fit(R[tr], y[tr]).predict_proba(R[te])[:, 1]
    return auc(y, p)


res = []
for s in range(10):
    R, yy = amostra_peneira(5000, s)
    res.append({"errada": peneira_errada(R, yy, semente=s),
                "certa": peneira_certa(R, yy, semente=s)})
res = pd.DataFrame(res)
print(res.round(3))
print("medianas:", res.median().round(3).to_dict())

**A coluna do futuro.** Imagine que a base trouxesse uma coluna `cobranca_nov`: o
cliente foi enviado à cobrança em novembro. Ela é registrada **depois** do calote de
outubro — vale 1 para quem deu calote, com erro de registro em 10% dos casos.

In [ ]:
# Bloco 4 — a coluna do futuro. COMPLETE.

def coluna_cobranca(y, erro=0.10, semente=11):
    """Uma coluna que só existe DEPOIS do calote: o cliente foi enviado à
    cobrança em novembro. Vale 1 para quem deu calote em outubro, com erro de
    registro em 10% dos clientes."""
    rng = np.random.default_rng(semente)
    troca = rng.random(len(y)) < erro
    return np.where(troca, 1 - y, y)


# COMPLETE: X com a coluna nova e a validacao do boosting nas mesmas DOBRAS
X_cobranca = ___
prob_c, _ = ___
print(f"boosting sem a coluna: AUC {auc(y, prob['boosting']):.3f}")
print(f"boosting com a coluna: AUC {auc(y, prob_c['boosting']):.3f}")

### Responda

> A validação cruzada foi feita do jeito certo, e a AUC subiu muito. O que está errado? Que
> pergunta, feita a cada coluna, teria evitado o problema?

## Bloco 5 — tempo e painel

Um banco acompanha a inadimplência do crédito pessoal em 150 municípios, mês a mês, por
cinco anos. Ele quer prever a inadimplência **do mês que vem** nos municípios onde já
atua e a de **municípios novos** onde pensa entrar. O painel é simulado: a verdade é
conhecida, e dá para medir o erro que cada modelo comete de fato em cada uso.

In [ ]:
# Bloco 5 — o painel simulado. Leia, nao precisa alterar.
#
# Inadimplencia (%) do municipio i no mes t:
#   4 + NIVEL do municipio (-1,2 x log da renda relativa + uma parte que
#   nenhuma coluna mostra) + CICLO comum a todos (volta a media, rho = 0,9)
#   + 0,10 por ponto de desemprego acima de 9% no mes anterior
#   + 0,08 por ponto de Selic acima de 11% no mes anterior + ruido.
# O modelo ve renda, populacao, desemprego e Selic do mes anterior. A Selic e
# a mesma para todos os municipios: identifica o mes. Renda e populacao
# identificam o municipio.

N_MUNICIPIOS = 150
N_MESES = 60
RHO = 0.9
DP_CICLO = 0.3
DP_NIVEL = 0.6
DP_RUIDO = 0.4
COLUNAS_PAINEL = ['renda', 'populacao', 'desemprego', 'selic']
ORIGEM_TEMPO = 36
SEMENTE_PAINEL = 1


def mundo(semente=SEMENTE_PAINEL, meses=N_MESES + 13):
    """O ciclo e a Selic de cada mês. Gera 13 meses além dos 60, para o
    futuro."""
    rng = np.random.default_rng(semente)
    ciclo = np.zeros(meses + 1)
    for t in range(1, meses + 1):
        ciclo[t] = RHO * ciclo[t - 1] + rng.normal(0, DP_CICLO)
    selic = np.clip(11 + np.cumsum(rng.normal(0, 0.35, meses + 1)), 2, 16)
    return {"ciclo": ciclo, "selic": selic, "rng": rng}


def municipios(rng, n):
    renda = np.round(np.exp(rng.normal(np.log(1.2), 0.35, n)), 2)       # R$ mil
    populacao = np.round(np.exp(rng.normal(10.3, 1.0, n)))
    nivel = -1.2 * np.log(renda / 1.2) + rng.normal(0, DP_NIVEL, n)
    desemprego_base = np.clip(rng.normal(9, 3, n), 2, 25)
    return {"renda": renda, "populacao": populacao, "nivel": nivel,
            "desemprego_base": desemprego_base}


def inadimplencia(M, ciclo_t, desemprego, selic, ruido):
    return 4 + M["nivel"] + ciclo_t + 0.10 * (desemprego - 9) + 0.08 * (selic - 11) + ruido


def painel(W, M, meses):
    """Uma linha por município e mês. As colunas do mês t trazem o desemprego
    e a Selic do mês anterior, que é o que se conhece quando se prevê t."""
    rng = W["rng"]
    n = len(M["renda"])
    partes = []
    for t in meses:
        des = np.clip(M["desemprego_base"] + 2.0 * W["ciclo"][t - 1] + rng.normal(0, 0.8, n),
                      1, 35).round(1)
        selic = round(float(W["selic"][t - 1]), 2)
        y = inadimplencia(M, W["ciclo"][t], des, selic, rng.normal(0, DP_RUIDO, n))
        partes.append(pd.DataFrame({
            "municipio": np.arange(n), "mes": t, "renda": M["renda"],
            "populacao": M["populacao"], "desemprego": des, "selic": selic,
            "inadimplencia": y.round(2)}))
    return pd.concat(partes, ignore_index=True)


def painel_simulado(semente=SEMENTE_PAINEL):
    """Os 150 municípios nos 60 meses, o mundo que os gerou e os municípios."""
    W = mundo(semente)
    M = municipios(W["rng"], N_MUNICIPIOS)
    return painel(W, M, range(1, N_MESES + 1)), W, M


def modelo_linear():
    return LinearRegression()


def modelo_boosting_reg():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0)


def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))


def validacao_embaralhada(fabrica, D):
    """Cinco dobras sorteadas linha a linha, como se cada linha fosse
    independente das outras."""
    X, y = D[COLUNAS_PAINEL], D["inadimplencia"].to_numpy()
    p = np.zeros(len(y))
    for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
        p[te] = fabrica().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    return rmse(y - p)


def verdade_municipio_novo(fabrica, D, W, n_novos=1500):
    """O erro de verdade em municípios que o modelo nunca viu, nos mesmos 60
    meses: ajusta com os 150 e prevê 1.500 municípios novos."""
    rng = np.random.default_rng(99)
    Mn = municipios(rng, n_novos)
    Wn = dict(W, rng=rng)
    Dn = painel(Wn, Mn, range(1, N_MESES + 1))
    m = fabrica().fit(D[COLUNAS_PAINEL], D["inadimplencia"])
    return rmse(Dn["inadimplencia"].to_numpy() - m.predict(Dn[COLUNAS_PAINEL]))


def verdade_mes_novo(fabrica, D, W, M, futuros=40, meses=12):
    """O erro de verdade nos 12 meses seguintes aos 60, nos mesmos
    municípios: ajusta com os 60 meses e prevê cada mês futuro com o
    desemprego e a Selic do mês anterior. Média de 40 futuros possíveis do
    mesmo mundo (o ciclo e o ruído de cada um são sorteados)."""
    m = fabrica().fit(D[COLUNAS_PAINEL], D["inadimplencia"])
    erros = []
    for f in range(futuros):
        rng = np.random.default_rng(1000 + f)
        ciclo = list(W["ciclo"][:N_MESES + 1])
        selic = list(W["selic"][:N_MESES + 1])
        for _ in range(meses):
            ciclo.append(RHO * ciclo[-1] + rng.normal(0, DP_CICLO))
            selic.append(float(np.clip(selic[-1] + rng.normal(0, 0.35), 2, 16)))
        Wf = {"ciclo": np.array(ciclo), "selic": np.array(selic), "rng": rng}
        Df = painel(Wf, M, range(N_MESES + 1, N_MESES + meses + 1))
        erros.append(Df["inadimplencia"].to_numpy() - m.predict(Df[COLUNAS_PAINEL]))
    return rmse(np.concatenate(erros))


MODELOS_PAINEL = {"linear": modelo_linear, "boosting": modelo_boosting_reg}
painel_df, MUNDO, MUNICIPIOS = painel_simulado()
print(f"{painel_df['municipio'].nunique()} municipios, {painel_df['mes'].nunique()} meses, "
      f"{len(painel_df)} linhas")
painel_df.head()

In [ ]:
# Bloco 5 — um municipio no tempo e um mes entre os municipios. Rode.
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for m in (0, 1, 2):
    d = painel_df[painel_df["municipio"] == m]
    axs[0].plot(d["mes"], d["inadimplencia"], label=f"municipio {m}")
axs[0].plot(range(1, N_MESES + 1), 4 + MUNDO["ciclo"][1:N_MESES + 1], "k--", lw=1,
            label="ciclo comum (+ 4)")
axs[0].set_xlabel("mes"); axs[0].set_ylabel("inadimplencia (%)"); axs[0].legend()
d = painel_df[painel_df["mes"] == 30]
axs[1].scatter(d["renda"], d["inadimplencia"], s=14, color=AZUL)
axs[1].set_xlabel("renda media (R$ mil)"); axs[1].set_title("mes 30, os 150 municipios")
plt.show()

In [ ]:
# Bloco 5 — tres validacoes. COMPLETE.
#
# validacao_embaralhada ja esta pronta. Escreva as outras duas:
#   por municipio: GroupKFold(5), com os grupos dados pela coluna "municipio";
#   no tempo: para cada mes t de ORIGEM_TEMPO + 1 ate o ultimo, ajusta com os
#   meses anteriores a t e preve t (janela expansiva).


def validacao_por_municipio(fabrica, D):
    X, y = D[COLUNAS_PAINEL], D["inadimplencia"].to_numpy()
    p = np.zeros(len(y))
    # COMPLETE
    for tr, te in ___:
        p[te] = fabrica().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    return rmse(y - p)


def validacao_no_tempo(fabrica, D, origem=ORIGEM_TEMPO):
    X, y = D[COLUNAS_PAINEL], D["inadimplencia"].to_numpy()
    erros = []
    for t in range(origem + 1, int(D["mes"].max()) + 1):
        # COMPLETE
        tr, te = ___, ___
        erros.append(y[te] - fabrica().fit(X[tr], y[tr]).predict(X[te]))
    return rmse(np.concatenate(erros))


resultado = {}
for nome, fabrica in MODELOS_PAINEL.items():
    resultado[nome] = {
        "embaralhada": validacao_embaralhada(fabrica, painel_df),
        "por_municipio": validacao_por_municipio(fabrica, painel_df),
        "no_tempo": validacao_no_tempo(fabrica, painel_df),
        "verdade_municipio_novo": verdade_municipio_novo(fabrica, painel_df, MUNDO),
        "verdade_mes_novo": verdade_mes_novo(fabrica, painel_df, MUNDO, MUNICIPIOS),
    }
print(pd.DataFrame(resultado).T.round(3))

### ✔ CHECKPOINT II

O banco quer prever a inadimplência do mês que vem e escolher municípios novos onde
entrar. Na validação embaralhada, o boosting erra bem menos que o linear.

Em duas frases: que validação você mostraria à diretoria para cada uso, e que modelo
recomendaria em cada um? Por que o número da embaralhada não serve para nenhum dos dois?

## O laudo de validação

A tabela dos seis modelos, nas mesmas cinco dobras. Rode a célula e escreva, abaixo dela,
a recomendação: que modelo o banco deveria usar para aprovar crédito, e por quê — em
três a cinco frases, citando a métrica que sustenta cada argumento.

In [ ]:
# O laudo. Rode.
laudo = pd.DataFrame({
    nome: {"AUC": auc(y, p), "AUC: menor dobra": notas_dobra[nome]["auc"].min(),
           "AUC: maior dobra": notas_dobra[nome]["auc"].max(), "Brier": brier(y, p),
           "lucro com a regra de 20%": lucro[nome]["regra_20"]}
    for nome, p in prob.items()}).T.sort_values("AUC", ascending=False)
laudo.round(3)

**Recomendação:**

_(escreva aqui)_